# 📝 LangChain 기본 구조 과제 LV2 정답 — 체인 잇기·Runnable·기록 (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../../day18_LangChain_기본구조/data/` 입니다. 순수 함수는 결정적으로, 모델 출력은 타입·구조로 채점합니다.

아래 준비 셀을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day18_LangChain_기본구조/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이 과제에서 공통으로 쓰는 부품들 — 실행하면 준비 끝입니다.
#   교안_02 에서 배운 Runnable 3종을 한 번에 불러 둡니다(문제마다 다시 import 하지 않게).
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
parser = StrOutputParser()
print('부품 준비 완료 —', type(model).__name__, '+', type(parser).__name__)

## 1. 2단 체인 — 생성한 문구를 요약하기
**배경**: 먼저 상세한 판매 문구를 만들고, 그것을 한 문장으로 요약하는 2단 흐름을 만듭니다.

**요구사항**:
- **1단계 체인** `desc_chain`: `('human', '{name} (특징: {note}) 의 중고 판매 상세 설명을 3~4문장으로 써줘.')` 프롬프트에 `model | parser` 를 이어 만드세요.
- **2단계 체인** `sum_chain`: `('human', '다음 글을 한 문장으로 요약해줘:\n{draft}')` 프롬프트에 `model | parser` 를 이어 만드세요.
- `desc_chain.invoke({'name': '미러리스 카메라', 'note': '셔터수 적음, 렌즈 2종, 가방 포함'})` 결과를 변수 **`draft1`** 에, 그것을 `sum_chain.invoke({'draft': draft1})` 에 넣은 결과를 변수 **`summary1`** 에 담으세요.

**예시**: `draft1`·`summary1` 모두 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 체인을 각각 만들고, 앞 체인의 문자열 출력을 뒤 체인의 draft 변수로 넣는다.

세부구현:
1. desc_chain = 상세설명 프롬프트 | model | parser.
2. sum_chain = 요약 프롬프트 | model | parser.
3. 1단계 체인을 먼저 실행해 초안을 변수에 담고, 그 변수를 2단계 체인의 draft 값으로 넘긴다.
```

</details>

In [ ]:
desc_chain = ChatPromptTemplate.from_messages([
    ('human', '{name} (특징: {note}) 의 중고 판매 상세 설명을 3~4문장으로 써줘.'),
]) | model | parser
sum_chain = ChatPromptTemplate.from_messages([
    ('human', '다음 글을 한 문장으로 요약해줘:\n{draft}'),
]) | model | parser

# 앞 체인은 파서로 끝나 '문자열'을 낸다 — 그래서 뒤 프롬프트의 {draft} 자리에 그대로 넣을 수 있다.
draft1 = desc_chain.invoke({'name': '미러리스 카메라', 'note': '셔터수 적음, 렌즈 2종, 가방 포함'})
summary1 = sum_chain.invoke({'draft': draft1})
print('[초안]', draft1)
print('[요약]', summary1)

In [ ]:
# [자가채점]
assert set(desc_chain.steps[0].input_variables) == {'name', 'note'}
assert set(sum_chain.steps[0].input_variables) == {'draft'}   # 앞 결과를 받을 변수 이름
assert isinstance(draft1, str) and len(draft1.strip()) > 0
assert isinstance(summary1, str) and len(summary1.strip()) > 0
print(f'초안 {len(draft1)}자 → 요약 {len(summary1)}자')   # 길이는 참고만(모델 출력이라 채점하지 않는다)
print('✅ 통과!')

**해설**: 앞 체인의 **문자열 출력**을 뒤 체인의 `{draft}` 변수로 넘겨 2단 흐름을 만듭니다. 단계를 나누면 각 단계가 한 가지 일만 해서 다루기 쉽습니다.

## 2. 두 체인을 하나로 — 다리 부품 놓기
**배경**: 1번은 `invoke` 를 **두 번** 불러 손으로 이었습니다. 앞 체인이 내는 **문자열**을 뒤 프롬프트가 원하는 **딕셔너리**로 바꿔 주는 **다리 부품**을 사이에 끼우면, 두 체인이 **하나**가 되어 `invoke` 한 번으로 끝납니다.

**요구사항**:
- **말투 변환 체인** `polite_chain`: `('human', '다음 글을 아주 정중하고 친절한 말투로 다시 써줘:\n{draft}')` 프롬프트에 `model | parser` 를 이어 만드세요.
- 문자열 하나를 받아 `{'draft': 그 문자열}` 로 바꿔 주는 다리 부품을 `RunnableLambda` 로 만들어 변수 **`to_draft`** 에 담으세요.
- 1번의 `desc_chain` 과 이어 **`polite_flow = desc_chain | to_draft | polite_chain`** 를 만드세요.
- `polite_flow.invoke({'name': '미러리스 카메라', 'note': '셔터수 적음, 렌즈 2종, 가방 포함'})` 를 **한 번만** 불러 결과를 변수 **`polite2`** 에 담으세요.

**예시**: `to_draft.invoke('테스트')` → `{'draft': '테스트'}` / `polite2` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 체인의 출력(문자열)과 뒤 프롬프트가 원하는 입력(딕셔너리)의 모양 차이를 다리 부품이 메운다.

세부구현:
1. polite_chain = 말투변환 프롬프트 | model | parser.
2. to_draft = RunnableLambda(문자열을 {'draft': 문자열} 로 바꾸는 함수).
3. polite_flow = desc_chain | to_draft | polite_chain 로 잇고 한 번만 invoke 한다.
```

</details>

In [ ]:
polite_chain = ChatPromptTemplate.from_messages([
    ('human', '다음 글을 아주 정중하고 친절한 말투로 다시 써줘:\n{draft}'),
]) | model | parser

# 다리는 값을 바꾸는 부품이 아니라 '모양'을 맞추는 부품 — 키 이름이 뒤 프롬프트의 변수와 같아야 한다.
to_draft = RunnableLambda(lambda text: {'draft': text})

# 이제 셋이 한 체인이라 invoke 는 한 번뿐이다(1번에서는 두 번 불렀다).
polite_flow = desc_chain | to_draft | polite_chain
polite2 = polite_flow.invoke({'name': '미러리스 카메라', 'note': '셔터수 적음, 렌즈 2종, 가방 포함'})
print(polite2)

In [ ]:
# [자가채점]
assert to_draft.invoke('테스트') == {'draft': '테스트'}   # 다리 부품이 모양을 바꾸는지
# 체인에 체인을 이으면 안쪽이 펼쳐진다 — (프롬프트,모델,파서) + 다리 + (프롬프트,모델,파서) = 7
assert len(polite_flow.steps) == 7
assert isinstance(polite_flow.steps[3], RunnableLambda)   # 한가운데가 다리 부품
assert isinstance(polite2, str) and len(polite2.strip()) > 0
print('✅ 통과!')

**해설**: 앞 체인은 파서로 끝나 **문자열**을 내는데 뒤 프롬프트는 `{draft}` 를 채울 **딕셔너리**를 원합니다. 그 사이를 `RunnableLambda(lambda text: {'draft': text})` 가 메워 **모양**을 맞춰 주면 셋이 한 체인이 됩니다. 하나가 된 체인은 다시 부품이라 `batch` 로 여러 매물을 한꺼번에 돌릴 수도 있습니다 — 1번처럼 손으로 이으면 그렇게 못 합니다.

> 자가채점이 `steps` 를 **7개**로 확인하는 이유: 체인에 체인을 이으면 LangChain 이 **안쪽 체인을 펼쳐** 하나의 평평한 목록으로 만듭니다. `(프롬프트·모델·파서) + 다리 + (프롬프트·모델·파서)` = 7 이 되는 것이지, '체인 3개' 로 세지 않습니다.

## 3. 내 함수를 부품으로 — 연락처 마스킹
**배경**: 중고 거래 글에서 **개인 연락 정보**가 그대로 노출되지 않게 가리는 전처리 함수를 만들고, `RunnableLambda` 로 체인 부품이 될 수 있게 감쌉니다. 이 함수는 모델과 무관한 **순수 함수**입니다.

**요구사항**:
- 함수 **`mask_contact(text: str) -> str`** 를 만드세요. 아래 **금지 단어 리스트** 각각을 `'***'` 로 바꿔 돌려줍니다.
- 금지 단어: `['010', '카카오', '계좌번호']`
- 만든 함수를 `RunnableLambda` 로 감싼 부품 **`mask_step`** 을 만드세요.
- 그 부품을 **체인 앞단에 실제로 끼워** 보세요 — `mask_step` 뒤에 `('human', '다음 중고 거래 글을 한 문장으로 정리해줘:\n{input}')` 프롬프트를 이으려면 문자열을 `{'input': 문자열}` 로 바꾸는 다리(2번의 `to_draft` 와 같은 요령)가 하나 더 필요합니다. 완성한 체인을 **`safe_chain`** 에 담고, `POST` 를 넣은 결과를 **`safe3`** 에 담으세요.

**예시**: `mask_contact('연락은 010 또는 카카오로 주세요')` → `'연락은 *** 또는 ***로 주세요'` / `safe3` 는 비어 있지 않은 문자열이고, 그 안에 금지 단어가 남아 있지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 금지 단어 리스트를 돌며 text 안의 각 단어를 '***' 로 바꾼다(문자열 replace).

세부구현:
1. 금지 단어 리스트를 for 로 돌면서, 문자열 치환 메서드로 그 단어를 별표로 바꾼 결과를 다시 담는다.
2. 다 돌고 난 문자열을 반환한다.
3. RunnableLambda(mask_contact) 를 mask_step 에 담는다.
4. mask_step | (문자열을 {'input': 문자열} 로 바꾸는 부품) | 프롬프트 | model | parser 로 safe_chain 을 만든다.
5. safe_chain.invoke(POST) 를 safe3 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 마스킹 대상 중고 거래 글
POST = '미러리스 카메라 팝니다. 010 으로 연락 주시거나 카카오로 문의 주세요. 입금은 계좌번호 알려 드립니다.'

In [ ]:
def mask_contact(text):
    """금지 단어(연락처·계좌 등)를 '***' 로 가려 돌려준다."""
    banned = ['010', '카카오', '계좌번호']
    for word in banned:
        text = text.replace(word, '***')
    return text

mask_step = RunnableLambda(mask_contact)

# 감싸 두기만 하면 쓸모가 없다 — 체인 앞단에 실제로 끼워야 '전처리 부품'이 된다.
#   mask_step 이 내는 것은 문자열이므로, 프롬프트 앞에 모양을 맞추는 다리가 하나 더 필요하다.
clean_prompt = ChatPromptTemplate.from_messages([
    ('human', '다음 중고 거래 글을 한 문장으로 정리해줘:\n{input}'),
])
safe_chain = mask_step | RunnableLambda(lambda t: {'input': t}) | clean_prompt | model | parser
safe3 = safe_chain.invoke(POST)
print('가린 글:', mask_step.invoke(POST))
print('정리   :', safe3)

In [ ]:
# [자가채점]
assert mask_contact('연락은 010 또는 카카오로 주세요') == '연락은 *** 또는 ***로 주세요'
assert mask_contact('계좌번호로 입금해 주세요') == '***로 입금해 주세요'
assert mask_step.invoke('010 카카오') == '*** ***'   # RunnableLambda 로 감싼 부품도 같은 결과
assert len(safe_chain.steps) == 5   # 마스킹 | 다리 | 프롬프트 | 모델 | 파서
assert isinstance(safe3, str) and len(safe3.strip()) > 0
# 모델에 넘어가기 '전'의 문자열로 채점한다 — 모델 출력은 실행마다 달라 채점 근거가 될 수 없다.
assert not any(w in mask_step.invoke(POST) for w in ['010', '카카오', '계좌번호'])
print('✅ 통과!')

**해설**: 전처리는 모델과 무관한 **순수 함수**라 결과가 항상 일정합니다. 핵심은 그 함수를 `RunnableLambda` 로 감싸 **체인 앞단에 실제로 끼웠다**는 것입니다 — 감싸 두기만 하고 쓰지 않으면 그냥 함수와 다를 바가 없습니다. 이렇게 두면 **모델에 넘어가기 전에** 개인정보가 걸러집니다.

> 다만 이 방식은 정해 둔 낱말만 가립니다. `'010'` 처럼 짧은 숫자열은 `'2010년'` 같은 엉뚱한 곳까지 가려 버리기도 합니다. 실무에서는 **패턴**(정규표현식)이나 전용 도구를 쓰는데, 여기서는 '부품으로 끼운다' 는 구조에 집중합니다.

## 4. 동시 처리 — 요약과 감정을 한 번에
**배경**: 거래 후기 하나를 받아 **요약**과 **감정 판단**을 동시에 수행합니다.

**요구사항**:
- **요약 체인** `sm_chain`: `('human', '다음 후기를 한 문장으로 요약해줘:\n{input}')` + `model | parser`.
- **감정 체인** `st_chain`: `('human', '다음 후기의 감정을 긍정/부정/중립 중 하나로만 답해줘:\n{input}')` + `model | parser`.
- `RunnableParallel` 로 `summary=sm_chain`, `sentiment=st_chain` 두 갈래를 묶어 **`analyze4`** 를 만드세요.
- `analyze4.invoke(후기문장)` 결과를 변수 **`result4`** 에 담으세요. 후기 문장은 준비 셀이 준 `REVIEW` 를 그대로 쓰세요.

**예시**: `result4` 는 `summary`·`sentiment` 두 키를 가진 딕셔너리이고, 각 값은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 체인을 만들고 RunnableParallel 로 묶은 뒤 후기 문장을 invoke 한다.

세부구현:
1. sm_chain·st_chain 을 각각 프롬프트 | model | parser 로 만든다.
2. RunnableParallel(summary=sm_chain, sentiment=st_chain) 을 analyze4 에 담는다.
3. analyze4.invoke(REVIEW) 를 result4 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 채점에 쓰는 후기 문장
REVIEW = '거래는 약속 장소에서 빠르게 끝났고 물건 상태도 사진과 똑같아 만족스러웠어요. 다만 충전기가 정품이 아니라 조금 아쉬웠습니다.'

In [ ]:
sm_chain = ChatPromptTemplate.from_messages([
    ('human', '다음 후기를 한 문장으로 요약해줘:\n{input}'),
]) | model | parser
st_chain = ChatPromptTemplate.from_messages([
    ('human', '다음 후기의 감정을 긍정/부정/중립 중 하나로만 답해줘:\n{input}'),
]) | model | parser

# 같은 입력이 두 갈래에 동시에 들어가고, 결과는 키로 구분된 딕셔너리 하나로 모인다.
analyze4 = RunnableParallel(summary=sm_chain, sentiment=st_chain)
result4 = analyze4.invoke(REVIEW)
print('요약:', result4['summary'])
print('감정:', result4['sentiment'])

In [ ]:
# [자가채점]
# steps__ 는 RunnableParallel 이 들고 있는 '갈래 이름 -> 부품' 묶음이다.
assert set(analyze4.steps__) == {'summary', 'sentiment'}   # 두 갈래를 실제로 묶었는지
assert set(result4.keys()) == {'summary', 'sentiment'}
assert isinstance(result4['summary'], str) and len(result4['summary'].strip()) > 0
assert isinstance(result4['sentiment'], str) and len(result4['sentiment'].strip()) > 0
print('✅ 통과!')

**해설**: `RunnableParallel` 은 같은 입력을 여러 부품에 **동시에** 넣고 결과를 **딕셔너리**로 모읍니다. 서로 다른 분석을 한 번에 받아 뒷단계에서 함께 쓰기 좋습니다.

## 5. 대화 기록 누적
**배경**: 대화가 이어지면 기록에 질문과 답을 한 턴씩 쌓아야 합니다. 이 누적을 **순수 함수**로 만듭니다.

**요구사항**:
- 함수 **`add_turn(history, user_text, ai_text)`** 를 만드세요. 기록 리스트 뒤에 `('user', user_text)` 와 `('assistant', ai_text)` 를 차례로 붙인 **새 리스트**를 돌려줍니다.

**예시**: `add_turn([], '안녕', '반가워요')` → `[('user', '안녕'), ('assistant', '반가워요')]`

<details><summary>힌트</summary>

```text
접근방법:
- 기존 리스트에 두 튜플을 이어 붙인 새 리스트를 반환한다.

세부구현:
1. history + [('user', user_text), ('assistant', ai_text)] 를 반환한다.
```

</details>

In [ ]:
def add_turn(history, user_text, ai_text):
    """기록 뒤에 사용자 질문과 모델 답을 한 턴으로 붙인 새 리스트를 돌려준다."""
    # append 가 아니라 '+' 로 새 리스트를 만든다 — 원본이 조용히 바뀌는 사고를 막는다.
    #   순서도 중요하다: 사용자 말이 먼저, 모델 답이 뒤여야 다음 호출에서 대화가 자연스럽게 읽힌다.
    return history + [('user', user_text), ('assistant', ai_text)]

print(add_turn([], '안녕', '반가워요'))

In [ ]:
# [자가채점]
assert add_turn([], '안녕', '반가워요') == [('user', '안녕'), ('assistant', '반가워요')]
_h = add_turn([('user', 'a'), ('assistant', 'b')], 'c', 'd')
assert _h == [('user', 'a'), ('assistant', 'b'), ('user', 'c'), ('assistant', 'd')]
assert len(_h) == 4
print('✅ 통과!')

**해설**: 한 턴은 사용자 한 줄, 모델 한 줄로 **두 줄**입니다. `history + [...]` 로 새 리스트를 만들면 원본을 바꾸지 않아 안전합니다.

## 6. 최근 N턴만 남기기
**배경**: 기록이 무한정 길어지지 않게 최근 N턴만 남깁니다(5번과 짝 — 기록 관리의 다른 방향).

**요구사항**:
- 함수 **`keep_recent(history, n_turns)`** 를 만드세요. 기록에서 **최근 `n_turns` 턴**(= 뒤에서 `2 * n_turns` 줄)만 남긴 리스트를 돌려줍니다.

**예시**: 6줄짜리 기록에 `keep_recent(history, 2)` → 뒤 4줄만 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 한 턴이 두 줄이므로, 기록 리스트의 뒤쪽에서 턴 수의 두 배 만큼만 남긴다.

세부구현:
1. 리스트 슬라이싱으로 끝에서 (턴 수 x 2) 개 항목만 잘라 그대로 반환한다(음수 인덱스 활용).
```

</details>

In [ ]:
def keep_recent(history, n_turns):
    """기록에서 최근 n_turns 턴(2*n_turns 줄)만 남긴다."""
    # 한 턴이 두 줄이라 '뒤에서 2*n_turns 개' 를 남긴다.
    #   함정: n_turns 가 0 이면 -0 == 0 이라 history[0:] — 전부가 남는다(하나도 안 남는 게 아니다).
    return history[-2 * n_turns:]

sample = [('user', 'a'), ('assistant', 'b'), ('user', 'c'), ('assistant', 'd'),
          ('user', 'e'), ('assistant', 'f')]
print(keep_recent(sample, 2))

In [ ]:
# [자가채점]
sample = [('user', 'a'), ('assistant', 'b'), ('user', 'c'), ('assistant', 'd'),
          ('user', 'e'), ('assistant', 'f')]
assert keep_recent(sample, 2) == [('user', 'c'), ('assistant', 'd'), ('user', 'e'), ('assistant', 'f')]
assert keep_recent(sample, 1) == [('user', 'e'), ('assistant', 'f')]
assert len(keep_recent(sample, 1)) == 2
print('✅ 통과!')

**해설**: `history[-2*n:]` 는 리스트 뒤에서 `2*n` 개를 남깁니다. 한 턴이 두 줄이라 턴 수 N 은 줄 수 2N 에 대응합니다.

## 7. 기록을 넣은 대화 체인
**배경**: 5·6번의 기록을 프롬프트에 끼워, 앞 대화를 **기억**하는 대화 체인을 만듭니다.

**요구사항**:
- `[('system', '너는 중고 거래 상담원이다. 존댓말로 간결하게 답한다.'), MessagesPlaceholder('history'), ('human', '{input}')]` 로 프롬프트를 만들고 `model | parser` 를 이어 체인 **`chat7`** 을 만드세요.
- 아래 **제공된 기록** `hist7` 과 질문 `'그거 상태가 어때?'` 를 넣어 `chat7.invoke({'history': hist7, 'input': '그거 상태가 어때?'})` 한 결과를 변수 **`answer7`** 에 담으세요.

**예시**: `answer7` 은 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- system·MessagesPlaceholder·human 으로 프롬프트를 구성하고, history 와 input 을 함께 넘긴다.

세부구현:
1. 프롬프트에 MessagesPlaceholder('history') 를 넣는다.
2. chat7 = 프롬프트 | model | parser.
3. chat7.invoke({'history': hist7, 'input': 질문}) 를 answer7 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 채점에 쓰는 대화 기록
hist7 = [('user', '미러리스 카메라 팔아요'), ('assistant', '네, 미러리스 카메라 문의 주셨네요. 무엇을 도와드릴까요?')]

In [ ]:
chat7 = ChatPromptTemplate.from_messages([
    ('system', '너는 중고 거래 상담원이다. 존댓말로 간결하게 답한다.'),
    MessagesPlaceholder('history'),
    ('human', '{input}'),
]) | model | parser

# MessagesPlaceholder 자리에 hist7 이 끼워지므로, 모델이 '그거'를 앞 대화에서 찾아낸다.
answer7 = chat7.invoke({'history': hist7, 'input': '그거 상태가 어때?'})
print(answer7)

In [ ]:
# [자가채점]
assert set(chat7.steps[0].input_variables) == {'history', 'input'}   # 기록 자리를 실제로 뒀는지
assert isinstance(answer7, str) and len(answer7.strip()) > 0
print('✅ 통과!')

**해설**: `MessagesPlaceholder('history')` 자리에 앞 대화가 끼워지므로, 모델은 '그거'가 미러리스 카메라를 가리킴을 **기억**하고 답합니다. 이 구조가 다음 단원의 챗봇 뼈대가 됩니다.

## 8. 원본을 함께 남기기 — RunnablePassthrough
**배경**: 요약을 만들되 **원본 후기도 그대로** 함께 남기고 싶을 때가 있습니다. `RunnablePassthrough` 는 입력을 **가공 없이 통과**시키는 부품이라, `RunnableParallel` 의 한 갈래로 두면 원본과 요약을 **한 번에** 얻습니다(4번의 확장).

**요구사항**:
- **요약 체인** `sum_chain8`: `('human', '다음 후기를 한 문장으로 요약해줘:\n{input}')` + `model | parser`.
- `RunnableParallel` 로 `original=RunnablePassthrough()`, `summary=sum_chain8` 두 갈래를 묶어 **`keep_original`** 을 만드세요.
- `keep_original.invoke(후기문장)` 결과를 변수 **`result8`** 에 담으세요. 후기 문장은 준비 셀이 준 `REVIEW` 를 그대로 쓰세요.

**예시**: `result8` 은 `original`·`summary` 두 키를 가진 딕셔너리이고, `result8['original']` 은 원본 `REVIEW` 와 똑같고(통과), `result8['summary']` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 요약 체인과 RunnablePassthrough 를 RunnableParallel 의 두 갈래로 묶는다. 통과 갈래는 입력을 그대로 돌려준다.

세부구현:
1. sum_chain8 을 요약 프롬프트 | model | parser 로 만든다.
2. RunnableParallel(original=RunnablePassthrough(), summary=sum_chain8) 을 keep_original 에 담는다.
3. keep_original.invoke(REVIEW) 를 result8 에 담는다.
```

</details>

In [ ]:
sum_chain8 = ChatPromptTemplate.from_messages([
    ('human', '다음 후기를 한 문장으로 요약해줘:\n{input}'),
]) | model | parser

# Passthrough 갈래는 입력을 손대지 않고 통과시킨다 — 그래서 original 은 REVIEW 와 글자까지 같다.
keep_original = RunnableParallel(original=RunnablePassthrough(), summary=sum_chain8)
result8 = keep_original.invoke(REVIEW)
print('원본:', result8['original'][:20], '...')
print('요약:', result8['summary'])

In [ ]:
# [자가채점]
assert type(keep_original.steps__['original']).__name__ == 'RunnablePassthrough'
assert set(result8.keys()) == {'original', 'summary'}
assert result8['original'] == REVIEW   # RunnablePassthrough 는 입력을 그대로 통과시킨다(결정적)
assert isinstance(result8['summary'], str) and len(result8['summary'].strip()) > 0
print('✅ 통과!')

**해설**: `RunnablePassthrough()` 는 받은 입력을 **그대로** 돌려줍니다. 그래서 `original` 갈래는 원본 후기가 손대지 않고 통과하고(그래서 `== REVIEW` 로 정확히 채점 가능), `summary` 갈래는 요약을 만들어 둘을 한 딕셔너리로 모읍니다. 원본을 남겨 두면 뒷단계에서 요약과 원문을 함께 활용할 수 있습니다.

---
수고했어요! **2단 체인·RunnableLambda·RunnableParallel·RunnablePassthrough·대화 기록 관리**를 익혔습니다. LV3 에서는 이 부품들을 모아 상담 봇과 상품 설명 파이프라인을 완성합니다.